# Create GNAMPA Awards (INdAM - Gruppo Nazionale per l'Analisi Matematica, la Probabilità e le loro Applicazioni)

Creates INdAM-GNAMPA research-project awards ("Progetti di Ricerca GNAMPA")
from the group's own yearly funded-project lists, published as PDFs on
https://www.altamatematica.it/gnampa/attivita/progetti-di-ricerca/
(PROGETTI GNAMPA 2012 ... 2026; no 2021 call - the 2020 list, approved
19/02/2020, was carried into 2021). Projects last one calendar year and fund
the missions of a coordinator + 3-8 participants. **833 projects, 2012-2026,
100% title/coordinator, 99% coordinator institution, 30% amount (amounts are
printed only for 2022-2025: EUR 2,000-4,500 each, EUR 810,600 total)**
(local run 2026-10-01).

GNAMPA is one of INdAM's national research groups and has its own OpenAlex
funder row; INdAM's own fellowships/grants go under INdAM (F4320311030), not here.

**Prerequisites:**
- Run `scripts/local/gnampa_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/gnampa/gnampa_projects.parquet` (§1.4 shrink guard).

**Parsing (script):** ruled-table years via pdfplumber's table finder;
text-only Excel exports (2015-2018) rebuilt from word positions (cells are
bottom-aligned, 2016 top-aligned). One-cell coordinator names are split using
the GNAMPA members lists (aderenti, COGNOME/NOME separate); 97-100% of
coordinators match a member. Where the PDF has no institution column
(2020, 2022-2026), the coordinator's institution comes from that year's
members list (`lead_institution_source`). Two 2013 titles are clipped in the
source PDF itself (first line missing) and ship as printed.

**`funder_award_id` (§2.1.1):** no per-project code is published. Citing works
write "GNAMPA 2020" / "INdAM-GNAMPA Project 2019" or the year's shared INdAM
CUP code (e.g. CUP E53C22001930001, the same CUP for every 2023 GNAMPA and
GNCS project), so there is nothing project-specific to match. Synthetic key:
`GNAMPA-{year}-{coordinator-family-slug}` (+ given-name slug when two
coordinators share a surname in a year). Collapses onto 0 citation shells.

**Dates:** start_year = end_year = project year; start/end dates NULL.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320333369`
- display_name: Gruppo Nazionale per l'Analisi Matematica, la Probabilità e le loro Applicazioni
- ror_id / doi: from `openalex.funders.funders` (ror 02ktjph70, doi 10.13039/100012740)

**Priority:** `557` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gnampa_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gnampa/gnampa_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.gnampa_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.gnampa_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gnampa_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320333369 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320333369;

## Step 2: Create GNAMPA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gnampa_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320333369  -- GNAMPA (INdAM)
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', 'IT',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_struct
    FROM openalex.awards.gnampa_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Progetti di Ricerca GNAMPA ', g.project_year) as funder_scheme,
    'indam_gnampa_progetti' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.project_year AS INT) as start_year,
    TRY_CAST(g.project_year AS INT) as end_year,
    g.lead_struct as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    ARRAY(g.lead_struct) as investigators,
    g.source_pdf as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM lead g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_gnampa_progetti' AND priority = 557;

-- Insert into openalex_awards_raw with priority.
-- Priority 557: direct-from-funder ingest of GNAMPA's own yearly project lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    557 as priority
FROM openalex.awards.gnampa_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.gnampa_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every key is the synthetic GNAMPA form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^GNAMPA-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^GNAMPA-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.gnampa_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.gnampa_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.gnampa_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.gnampa_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.gnampa_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/PI, ~99% institution, ~30% amount: 2022-2025 only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.gnampa_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_gnampa_progetti'
GROUP BY provenance, priority;